# UAPP — S669 external validation (D5 ensemble)

Train a 5-member D5 ensemble on T2837, evaluate on S669 (Pancotti et al. 2022). This is the cleanest available test of generalisation for the campaign's production model.

**What you need in Drive (`/content/drive/MyDrive/uapp_cache/`) before starting:**
- `t2837_embeddings_v2_650m.pt`  (~26 MB, ESM2-650M cache)
- `t2837_metadata.csv`
- `t2837_bio_features_650m_extended.pt`  (k=13 D5 features)

If any are missing, rebuild via the recipe in `REPORT.md` §9 before continuing.

**Path:** D5 only (k=13: RSA + chemistry + sequence-derived structural). No DSSP / AlphaFold downloads needed. D5 had the best Spearman in the campaign (0.348 vs D6's 0.331).

## 1. Environment

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf uapp
!git clone https://github.com/RoselindSi/uapp.git
%cd /content/uapp
!git log --oneline -5

In [ ]:
!pip install -q transformers torch numpy pandas tqdm scipy biopython

## 2. Restore T2837 caches from Drive

If any are missing, **stop here** and rebuild — the S669 eval needs them.

In [ ]:
import os, shutil
os.makedirs('cache', exist_ok=True)

needed = [
    't2837_embeddings_v2_650m.pt',
    't2837_metadata.csv',
    't2837_bio_features_650m_extended.pt',
]
missing = []
for f in needed:
    src = f'/content/drive/MyDrive/uapp_cache/{f}'
    dst = f'cache/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✓ {f}  ({os.path.getsize(dst)/1e6:.1f} MB)')
    else:
        print(f'✗ MISSING: {f}')
        missing.append(f)

assert not missing, f'Missing required caches: {missing}.  Rebuild before continuing.'

## 3. Get the S669 dataset

Authoritative source: Pancotti et al. 2022 on Zenodo (record 7568094). The zip contains `S669/S669.csv` plus WT and mutant PDB structures — we only need the CSV for D5.

In [ ]:
!mkdir -p data/s669
!wget -q -O data/s669/S669.zip https://zenodo.org/records/7568094/files/S669.zip
!unzip -o -q data/s669/S669.zip -d data/s669/
!ls data/s669/S669/
!head -3 data/s669/S669/S669.csv
!wc -l data/s669/S669/S669.csv

In [ ]:
# Manual-upload fallback (skip if previous cell worked)
# from google.colab import files
# uploaded = files.upload()
# import shutil, os
# fname = list(uploaded.keys())[0]
# os.makedirs('data/s669/S669', exist_ok=True)
# shutil.move(fname, 'data/s669/S669/S669.csv')
# !head -3 data/s669/S669/S669.csv && wc -l data/s669/S669/S669.csv

## 4. Convert S669 → T2837 metadata format

Auto-detects column names. If many rows are skipped as "no-WT-seq", pass `--fasta data/s669/wildtypes.fasta`. If column auto-detect fails, override with `--protein-col / --mut-col / --ddg-col`.

In [ ]:
!python scripts/17_s669_to_t2837_format.py \
    --s669-csv data/s669/S669/S669.csv \
    --out cache/s669_metadata.csv

## 5. Cache S669 ESM2-650M embeddings

~3 min on a T4. Script 01 enforces `max(1, ...)` per split, so a couple of rows may end up in `train` or `val` instead of `test` — script 18 picks the largest split automatically and warns.

In [ ]:
!python scripts/01_cache_embeddings_esm_v2.py \
    --t2837-csv cache/s669_metadata.csv \
    --out cache/s669_embeddings_650m.pt \
    --metadata-out cache/s669_metadata_processed.csv \
    --val-fraction 0 --test-fraction 1.0 \
    --esm-model facebook/esm2_t33_650M_UR50D \
    --device cuda --seed 42

## 6. Build S669 D5 bio features (k=13)

In [ ]:
!python scripts/06_build_bio_features.py \
    --metadata-csv cache/s669_metadata_processed.csv \
    --embeddings   cache/s669_embeddings_650m.pt \
    --out          cache/s669_bio_features_650m_extended.pt \
    --include-extended

## 7. Train D5 ensemble on T2837, evaluate on S669

~5 min on a T4. Trains 5 D5 heads on T2837 train, early-stops on T2837 val, predicts on:
- T2837 test — sanity check, should reproduce campaign numbers (RMSE ≈ 1.50, Spearman ≈ 0.348 within ± 0.07).
- S669 — the headline external-validation result.

In [ ]:
!python scripts/18_evaluate_on_s669.py \
    --t2837-emb cache/t2837_embeddings_v2_650m.pt \
    --t2837-bio cache/t2837_bio_features_650m_extended.pt \
    --s669-emb  cache/s669_embeddings_650m.pt \
    --s669-bio  cache/s669_bio_features_650m_extended.pt \
    --out       outputs/s669_eval_d5 \
    --ablation D5 --members 5 --device cuda

## 8. Save outputs to Drive

In [ ]:
!cp -r outputs/s669_eval_d5 /content/drive/MyDrive/uapp_cache/
!cp cache/s669_metadata.csv \
    cache/s669_metadata_processed.csv \
    cache/s669_embeddings_650m.pt \
    cache/s669_bio_features_650m_extended.pt \
    /content/drive/MyDrive/uapp_cache/
print('saved to Drive')

## 9. Inspect the result

In [ ]:
import json, pathlib
s = json.loads(pathlib.Path('outputs/s669_eval_d5/ensemble_summary.json').read_text())

print('=' * 70)
print('T2837 test ensemble  (sanity check vs campaign numbers)')
print('=' * 70)
for k in ('n_test', 'rmse', 'mae', 'nll', 'ice', 'spearman',
         'cov@0.90', 'cov@0.95', 'top0.20'):
    print(f'  {k:<12} = {s["t2837_test"]["ensemble"][k]}')

print()
print('=' * 70)
print('S669 ensemble  (headline external validation)')
print('=' * 70)
for k in ('n_test', 'rmse', 'mae', 'nll', 'ice', 'spearman',
         'cov@0.90', 'cov@0.95', 'top0.20'):
    print(f'  {k:<12} = {s["s669"]["ensemble"][k]}')

print()
print('=' * 70)
print('Cross-dataset Δ (S669 − T2837)')
print('=' * 70)
te = s['t2837_test']['ensemble']; se = s['s669']['ensemble']
for k in ('rmse', 'nll', 'ice', 'spearman'):
    print(f'  Δ {k:<10} = {se[k] - te[k]:+.4f}')

## Decision rule

| S669 ensemble Spearman | Verdict | Action |
|---|---|---|
| ≥ 0.30 | Strong generalisation | Add S669 row to `REPORT.md` deliverable table; unstrike Limitation #5 |
| 0.20–0.30 | Real but degraded | Document the drop honestly; still a publishable finding |
| < 0.20 | Generalisation fails | Document as negative result; investigate domain shift |

T2837 test sanity row should land within ~0.05 of (RMSE 1.50, Spearman 0.348). If it's wildly off, something's wrong with the cache or feature alignment — stop and check before trusting the S669 numbers.